In [ ]:
import json
import re

import anthropic
import chardet
from Bio import Entrez
from Bio import Medline
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

with open('./list_of_stains.json', 'r', encoding='utf-8') as file:
    stain_list = json.load(file)
len(stain_list)


In [ ]:
with open("stains_from_first_data_extraction.txt", "rb") as f:
    raw_data = f.read()
    result = chardet.detect(raw_data)
    encoding = result['encoding']
print(encoding)
stains_from_concat = raw_data.decode(encoding).splitlines()

stain_list.extend(stains_from_concat)
stain_list = list(set(stain_list))
len(stain_list)

In [ ]:
GREEK_CHAR_TO_LATIN_NAME = {
    'α': 'a', 'Α': 'Alpha',
    'β': 'B', 'Β': 'Beta',
    'γ': 'gamma', 'Γ': 'Gamma',
    'δ': 'delta', 'Δ': 'Delta',
    'ε': 'epsilon', 'Ε': 'Epsilon',
    'ζ': 'zeta', 'Ζ': 'Zeta',
    'η': 'eta', 'Η': 'Eta',
    'θ': 'theta', 'Θ': 'Theta',
    'ι': 'iota', 'Ι': 'Iota',
    'κ': 'kappa', 'Κ': 'Kappa',
    'λ': 'lambda', 'Λ': 'Lambda',
    'μ': 'mu', 'Μ': 'Mu',
    'ν': 'nu', 'Ν': 'Nu',
    'ξ': 'xi', 'Ξ': 'Xi',
    'ο': 'omicron', 'Ο': 'Omicron',
    'π': 'pi', 'Π': 'Pi',
    'ρ': 'rho', 'Ρ': 'Rho',
    'σ': 'sigma', 'Σ': 'Sigma',
    'τ': 'tau', 'Τ': 'Tau',
    'υ': 'upsilon', 'Υ': 'Upsilon',
    'φ': 'phi', 'Φ': 'Phi',
    'χ': 'chi', 'Χ': 'Chi',
    'ψ': 'psi', 'Ψ': 'Psi',
    'ω': 'omega', 'Ω': 'Omega',
}


def generate_variations(s: str) -> list[str]:
    """
    Generates a list of variations for a given string.

    This includes:
    1.  Replacing Greek characters (α, β) with Latin names (alpha, beta).
    2.  Adding/removing separators (dashes, spaces) between letters and digits.
    3.  Swapping existing dashes and spaces.
    4.  Providing a "cleaned" version with no separators.
    """
    # Use a set to automatically handle duplicate variations.
    variations = {s}

    # --- 1. Generate Latin Name Variations from Greek Chars ---
    # Create a new variation if any Greek characters are found and replaced.
    text_with_latin_names = s
    changed = False
    for greek, latin in GREEK_CHAR_TO_LATIN_NAME.items():
        if greek in text_with_latin_names:
            text_with_latin_names = text_with_latin_names.replace(greek, latin)
            changed = True

    if changed:
        variations.add(text_with_latin_names)

    # --- 2. Generate Separator Variations ---
    # This logic is applied to all base variations (original and the one with Latin names).
    for variation in list(variations):
        # Create a "cleaned" version by removing dashes and spaces.
        cleaned = variation.replace('-', '').replace(' ', '')
        variations.add(cleaned)

        # From the cleaned string, add variations with separators at boundaries.
        # Boundary: Non-Digit to Digit (\D\d)
        if re.search(r'(\D)(\d)', cleaned):
            variations.add(re.sub(r'(\D)(\d)', r'\1-\2', cleaned))
            variations.add(re.sub(r'(\D)(\d)', r'\1 \2', cleaned))

        # Boundary: Digit to Non-Digit (\d\D)
        if re.search(r'(\d)(\D)', cleaned):
            variations.add(re.sub(r'(\d)(\D)', r'\1-\2', cleaned))
            variations.add(re.sub(r'(\d)(\D)', r'\1 \2', cleaned))

        # Handle swapping existing separators in the original variation.
        if ' ' in variation:
            variations.add(variation.replace(' ', '-'))
        if '-' in variation:
            variations.add(variation.replace('-', ' '))

    return sorted(list(variations))

In [ ]:
stain_variation_dict = {stain: generate_variations(stain) for stain in stain_list}

In [ ]:
stain_variation_dict

In [ ]:
import pandas as pd
from Bio import Entrez
from tqdm import tqdm


Entrez.email = '****'
queries_results = []

for key, value in tqdm(stain_variation_dict.items()):
    current_stain_variations = ' OR '.join(f'"{item}"[tiab]' for item in value)

    positive_terms = (
        f'({current_stain_variations}) AND '
        '('
        '"Ovarian Neoplasms"[mh] OR '
        '"ovarian cancer"[tiab] OR '
        '"ovarian carcinoma"[tiab] OR '
        '"ovarian tumor"[tiab] OR '
        '"ovarian neoplasm"[tiab] OR '
        '"ovarian malignancy"[tiab] OR '
        '"ovarian malignancies"[tiab] OR '
        '"ovarian adenocarcinoma"[tiab] OR '
        '"ovarian epithelial tumor"[tiab] OR '
        '"ovarian mass"[tiab] OR '
        '"ovarian cystadenocarcinoma"[tiab]'
        ') AND '
        '('
        '"Immunohistochemistry"[MeSH] OR '
        '"immunohistochemistry"[tiab] OR '
        '"immuno-histochemistry"[tiab] OR '
        '"IHC"[tiab] OR '
        '"IHC staining"[tiab] OR '
        '"immunostaining"[tiab]'
        ') AND '
        '("1997/01/01"[Date - Create] : "3000"[Date - Create]) AND '
        '"English"[Language]'
    )

    # 2. Define all the terms you want to EXCLUDE
    negative_terms = (
        'NOT "Animals"[mh:noexp] '
        'NOT "Case Reports"[Publication Type] '
        'NOT ("case report"[tiab] OR "case series"[tiab]) '  # Note the parentheses here
        'NOT "reclassified"[tiab] '
        'NOT "misclassified"[tiab] '
        'NOT "initially"[tiab] '
        'NOT "Diagnostic Errors"[MeSH] '
        'NOT "demethyl*"[tiab]'
    )

    query = f"({positive_terms}) {negative_terms}"

    try:
        esearch_res = Entrez.esearch(db='pubmed', retmax=99999, term=query, retmode='xml')
        record = Entrez.read(esearch_res)
    except Exception as e:
        esearch_res = Entrez.esearch(db='pubmed', retmax=99999, term=query, retmode='xml')
        record = Entrez.read(esearch_res)

    result = ','.join(record['IdList'])
    result_dict = {'stain': key, 'result_no': len(record['IdList']), 'ids': result}
    queries_results.append(result_dict)